In [2]:
import re
import json

from selected_podcast_categories import PODCAST_CATEGORIES


def slugify(name):
    """Convert display name to slug format, replacing ' & ' with '_and_'."""
    name = name.strip().lower()
    name = name.replace(" & ", "_and_")
    name = re.sub(r"\s+", "_", name)
    return name


def transform_categories(podcast_categories):
    category_mappings = {}

    for top_category, subcat_list in podcast_categories.items():
        # Find the first item where display name matches top_category for displayName fallback
        fallback_display_name = None
        for parent, display in subcat_list:
            if parent == top_category and slugify(display) == top_category:
                fallback_display_name = display
                break
        # Default fallback if not found
        if fallback_display_name is None:
            fallback_display_name = subcat_list[0][1]

        subcategories = {}
        for pair in subcat_list:
            _, display = pair
            cat_slug = slugify(display)
            subcategories[cat_slug] = {
                "name": cat_slug,
                "displayName": display,
                "match_categories": [pair],
            }

        # Add "subcategory" default
        subcategories["subcategory"] = {
            "name": top_category,
            "displayName": fallback_display_name,
            "match_categories": [[top_category, fallback_display_name]],
        }

        category_mappings[top_category] = {
            "name": top_category,
            "displayName": fallback_display_name,
            "subcategories": subcategories,
        }

    return category_mappings


podcast_categories = PODCAST_CATEGORIES

new_categories = transform_categories(podcast_categories)

with open("new_category_mappings.json", "w", encoding="utf-8") as f:
    json.dump(new_categories, f)